In [1]:
import pandas as pd
import numpy as np

import re
import os

INPUT_PATH = "data.csv" 
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:
#Load the data
df = pd.read_csv(INPUT_PATH, encoding = 'ISO-8859-1', low_memory=False)
step0_count = len(df)
print(f"Step 0: Initial count of rows: {step0_count}")

Step 0: Initial count of rows: 541909


In [3]:
#Remove duplicates
#why? beacause duplicates can skew the analysis and lead to inaccurate insights. Removing duplicates ensures that each transaction is counted only once, providing a more accurate representation of the data.

dup_count  = df.duplicated().sum()
print(f"Step 1 - exact duplicates found: {dup_count}")

Step 1 - exact duplicates found: 5268


In [4]:
df = df.drop_duplicates().copy()
step1_count = len(df)
print(f"Step 1 - rows after removing duplicates: {step1_count}")


Step 1 - rows after removing duplicates: 536641


In [5]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/10 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/10 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/10 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/10 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/10 8:34,1.69,13047.0,United Kingdom


In [6]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

In [7]:
df["InvoiceDateTime"] = pd.to_datetime(
    df["InvoiceDate"], format="%m/%d/%y %H:%M", errors="coerce"
)

In [8]:

unparsed_dates = df["InvoiceDateTime"].isna().sum()
print(f"Step 2 - rows where date failed to parse: {unparsed_dates}")
if unparsed_dates > 0:
    # Don't silently drop these - keep them visible for inspection
    df[df["InvoiceDateTime"].isna()].to_csv(
        f"{OUTPUT_DIR}/unparsed_dates.csv", index=False
    )
    print(f"  -> saved to {OUTPUT_DIR}/unparsed_dates.csv for review")

Step 2 - rows where date failed to parse: 0


In [9]:
#flag cancelations / returns 

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df["IsCancelled"] = df["InvoiceNo"].str.startswith("C")
print(f"Step 3 - cancelled rows flagged: {df['IsCancelled'].sum()}")

Step 3 - cancelled rows flagged: 9251


In [10]:
#flag non-product transactions

product_pattern = re.compile(r"^\d{5}[A-Za-z]{0,2}$")
df["StockCode"] = df["StockCode"].astype(str)
df["IsProduct"] = df["StockCode"].apply(lambda x: bool(product_pattern.match(x)))

non_product_mask = (~df["IsProduct"]) & (~df["IsCancelled"])
print(f"Step 4 - non-product rows flagged: {non_product_mask.sum()}")
print("  Top non-product codes:")
print(df.loc[non_product_mask, "StockCode"].value_counts().head(10).to_string())

Step 4 - non-product rows flagged: 2406
  Top non-product codes:
StockCode
POST            1130
DOT              709
M                322
C2               142
DCGSSGIRL         13
BANK CHARGES      12
DCGSSBOY          11
gift_0001_20      10
gift_0001_10       9
gift_0001_30       8


In [11]:
#Split into side tables for easier analysis

returns_df = df[df["IsCancelled"]].copy()
non_product_df = df[non_product_mask].copy()

In [12]:
#candidates for cleaned_sales: rows that are not cancelled and are products

candidates = df[(~df["IsCancelled"]) & (df["IsProduct"])].copy()

#Determine exclusion reason for rows that still fail quality checks. This will help us understand why certain rows are being excluded from the cleaned dataset.
def exclusion_reason(row):
    if pd.isna(row["CustomerID"]):
        return "missing_customer_id"
    if pd.isna(row["Description"]) or str(row["Description"]).strip() == "":
        return "missing_description"
    if row["Quantity"] <= 0:
        return "non_positive_quantity"
    if row["UnitPrice"] <= 0:
        return "non_positive_unit_price"
    if pd.isna(row["InvoiceDateTime"]):
        return "unparsed_date"
    return None

candidates["exclusion_reason"] = candidates.apply(exclusion_reason, axis=1)
excluded_df = candidates[candidates["exclusion_reason"].notna()].copy()

print(f"Step 5 - rows exclusion reason breakdown: ")
print(excluded_df["exclusion_reason"].value_counts().to_string())

Step 5 - rows exclusion reason breakdown: 
exclusion_reason
missing_customer_id        133801
non_positive_unit_price        33


In [13]:
candidates.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateTime,IsCancelled,IsProduct,exclusion_reason
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/10 8:26,7.65,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/10 8:26,4.25,17850.0,United Kingdom,2010-12-01 08:26:00,False,True,NaN
7,536366,22633,HAND WARMER UNION JACK,6,12/1/10 8:28,1.85,17850.0,United Kingdom,2010-12-01 08:28:00,False,True,NaN
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/10 8:28,1.85,17850.0,United Kingdom,2010-12-01 08:28:00,False,True,NaN
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/10 8:34,1.69,13047.0,United Kingdom,2010-12-01 08:34:00,False,True,NaN


In [14]:
# build the cleaned dataset by excluding rows with any exclusion reason

cleaned = candidates[candidates["exclusion_reason"].isna()].copy()

cleaned["Description"] = cleaned["Description"].str.strip()
cleaned["CustomerID"] = cleaned["CustomerID"].astype(int)
cleaned["TotalPrice"] = (cleaned["Quantity"] * cleaned["UnitPrice"]).round(2)

cleaned = cleaned [[
        "InvoiceNo",
        "StockCode", 
        "Description",
        "Quantity",
        "InvoiceDateTime",
        "UnitPrice",
        "CustomerID",
        "Country",
        "TotalPrice"
]]


In [15]:
# Audit trail - confirm every row is accounted for in the final cleaned dataset, either as a valid row or as an excluded row with a reason.

summary = pd.DataFrame({
    "stage":[
        "raw_rows",
        "after_dedup",
        "cleaned_sales",
        "returns",
        "non_product_txn",
        "excluded_rows",
        "sum_of_buckets"
    ],
    "row_count":[
        step0_count,
        step1_count,
        len(cleaned),
        len(returns_df),
        len(non_product_df),
        len(excluded_df),
        len(cleaned) + len(returns_df) + len(non_product_df) + len(excluded_df)
    ]
})

print (f"\nStep 7 Audit trail:")
print (summary.to_string(index=False))

assert step1_count == len(cleaned) + len(returns_df) + len(non_product_df) + len(excluded_df), "Row counts don't reconcile! Check the filtering logic above."


Step 7 Audit trail:
          stage  row_count
       raw_rows     541909
    after_dedup     536641
  cleaned_sales     391150
        returns       9251
non_product_txn       2406
  excluded_rows     133834
 sum_of_buckets     536641


In [16]:
#Sanity Checks

print("\nStep 8 - sanity checks (all should be 0):")
print("  Quantity <= 0:      ", (cleaned["Quantity"] <= 0).sum())
print("  UnitPrice <= 0:     ", (cleaned["UnitPrice"] <= 0).sum())
print("  Missing CustomerID: ", cleaned["CustomerID"].isna().sum())
print("  Missing Description:", cleaned["Description"].isna().sum())
print("  Missing date:       ", cleaned["InvoiceDateTime"].isna().sum())
print("  Cancelled invoices: ", cleaned["InvoiceNo"].str.startswith("C").sum())


Step 8 - sanity checks (all should be 0):
  Quantity <= 0:       0
  UnitPrice <= 0:      0
  Missing CustomerID:  0
  Missing Description: 0
  Missing date:        0
  Cancelled invoices:  0


In [17]:
#Save the cleaned dataset to a CSV file for further analysis or reporting.

cleaned.to_csv(f"{OUTPUT_DIR}/cleaned_sales.csv", index=False)
returns_df.to_csv(f"{OUTPUT_DIR}/returns.csv", index=False)
non_product_df.to_csv(f"{OUTPUT_DIR}/non_product_transactions.csv", index=False)
excluded_df.to_csv(f"{OUTPUT_DIR}/excluded_rows.csv", index=False)
summary.to_csv(f"{OUTPUT_DIR}/audit_trail.csv", index=False)

print(f"\nDone. Files saved in ./{OUTPUT_DIR}")
print(f"Final cleaned_sales row count: {len(cleaned)}")


Done. Files saved in ./output
Final cleaned_sales row count: 391150
